# Re-identification review

**Pipeline step 9 of 10.** Follows 8 Harmonise and pool (`harmonise.ipynb`, `harmonise_checks.ipynb`) and comes before 10 Anonymisation. It reads the static table (step 5) and the harmonised files (step 8) and changes nothing. See `PIPELINE.md`.

**Question.** How identifiable are stays from the static fields and from which concepts each stay has? This covers unique stays and small cells on combinations of age, sex, ethnicity, diagnosis, weight and outcome, repeat patients, rare categories, values found at one site only, and how well the site can be guessed. Counts only.

**Cohort.** Stays over 6 hours, with GOSH stays from the Epic switch (2019-04-20) left out, because their data are incomplete. Nothing is dropped for privacy here: the stays that anonymisation deletes for being in a small cell are still present, so this review measures them. The count is therefore slightly higher than the release.

**What it feeds.** The coarsening and suppression rules that step 10 applies (`reid.py`, used by `anonymise.py`). Ethnicity and length of stay are kept in this review as evidence for dropping them from the release. The risk of the release key itself is measured at step 10.

**Output.** `reports/Reid_<date>.md` and the figures `reid_*.png` in `reports/`. Both stay in the SDE.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from datetime import date
from pathlib import Path
import numpy as np
import pandas as pd
open_picu_dir = Path("/home/jormond/Documents/DS217/users/jormond/open_picu")
sys.path.insert(0, str(open_picu_dir / "code"))
import harmonise as H, reid as R, diagnostics as D, static as ST, concepts as C

Paths, and the pooled static table limited to the analysis cohort: stays over 6 hours, without the GOSH stays from the Epic switch (2019-04-20) on. The printed count is the cohort for the whole review.

In [ ]:
derived_dir = open_picu_dir / "derived"
harmonised_dir = derived_dir / "harmonised"
reports_dir = open_picu_dir / "reports"

static = pd.read_parquet(derived_dir / "static" / "pooled_static.parquet")
co = H.cohort(static)
static = static.merge(co[["site", "stay_key"]], on=["site", "stay_key"])
epic = (static["site"] == "GOSH") & (pd.to_datetime(static["t0"]) >= pd.Timestamp(H.EPIC_SWITCH))
static = static[~epic].reset_index(drop=True)
print(len(static), "stays;", int(epic.sum()), "GOSH Epic-era stays left out")
static.groupby("site").size()

**Uniqueness.** Stays that are alone in their cell, and stays in cells smaller than 10, for each combination. Bands are in `reid.py`.

In [ ]:
uniq = R.uniqueness(static)
display(uniq)

**Repeat patients, rare categories and site-only values.**

In [ ]:
display(R.repeat_patients(static))
display(R.rare_categories(static))
display(R.site_only_values(static))

**Category lists for ethnicity and sex.** Counts per site, with counts under 10 shown as `<10`. Used to design the broad ethnicity groups and to see what the rare sex category is called.

In [ ]:
static_by_site = {s: g for s, g in static.groupby("site")}
eth_table = ST.category_table(static_by_site, "ethnicity")
sex_table = ST.category_table(static_by_site, "sex")

**Candidate Tier 2 coarsening.** No year, patient key replaced by `split_group`, `pim_version` dropped, rare sex values set to missing. Rare diagnosis groups are not merged here; small cells are handled by local suppression below. Two ethnicity options compared: six broad groups, or the recorded labels with rare ones merged. Counts only.

In [ ]:
coarse_g = R.coarsen(static, ethnicity="group")
coarse_d = R.coarsen(static, ethnicity="detail")
print(R.split_group_sizes(coarse_g))

res_g = R.uniqueness(coarse_g, combos=R.T2_COMBOS).assign(ethnicity="group")
res_d = R.uniqueness(coarse_d, combos=R.T2_COMBOS).assign(ethnicity="detail")
res = pd.concat([res_g, res_d])
display(res[res["site"] == "pooled"][["ethnicity", "combination", "unique_stays", "stays_in_cells_below_10", "share_below_k"]])
display(res[res["site"] != "pooled"][["ethnicity", "combination", "site", "share_below_k"]])

**Coarser bands and local suppression.** Coarser age, weight and length-of-stay bands with the six ethnicity groups, then values suppressed field by field for stays in cells under 10 (stays are all kept). The report shows how many values each field loses.

In [ ]:
coarse2 = R.coarsen(static, ethnicity="group", age_edges=R.AGE_COARSE, weight_edges=R.WEIGHT_COARSE, los_edges=R.LOS_COARSE)
display(R.uniqueness(coarse2, combos=R.T2_COMBOS).query("site == 'pooled'")[["combination", "unique_stays", "stays_in_cells_below_10", "share_below_k"]])

key = ["age_band", "sex", "ethnicity", "diagnosis_group", "died", "weight_band", "los_band"]
supp, rep = R.suppress_to_k(coarse2, key, ["los_band", "weight_band", "ethnicity", "diagnosis_group"])
display(rep)
display(R.uniqueness(supp, combos={"all": key}))

**Suppression without length of stay.** Length of stay is left out of the static Tier 2 fields: it is almost continuous, so it creates most of the small cells, and it is an outcome (future information) that users can compute from the length of the time series. The key is age band, sex, ethnicity group, diagnosis group, outcome and weight band. Fields are suppressed in the order weight, ethnicity, diagnosis, least valuable first. All stays are kept.

In [ ]:
key = ["age_band", "sex", "ethnicity", "diagnosis_group", "died", "weight_band"]
supp, rep2 = R.suppress_to_k(coarse2, key, ["weight_band", "ethnicity", "diagnosis_group"])
display(rep2)
display(R.uniqueness(supp, combos={"all": key}))
R.plot_suppression(rep2, len(coarse2), path=reports_dir / "reid_suppression_no_los.png");

**What is grouped.** The ethnicity groups with stays per site (counts under 10 shown as `<10`), and the diagnosis groups with the sites at which a group has fewer than 10 stays. Those small diagnosis groups are not merged; they are handled by local suppression.

In [ ]:
display(R.group_table(static, "ethnicity"))
display(R.merge_table(static, "diagnosis_group"))

**Figures for the presentation.** Identifiability as fields are added (default against coarse bands), the cost of local suppression, repeat patients, and ethnicity groups by site. Saved to `reports/`, which is git-ignored.

In [ ]:
variants = {"default bands": R.uniqueness(R.coarsen(static), combos=R.T2_COMBOS),
            "coarse bands": R.uniqueness(coarse2, combos=R.T2_COMBOS)}
R.plot_buildup(variants, path=reports_dir / "reid_buildup.png");
R.plot_suppression(rep, len(coarse2), path=reports_dir / "reid_suppression.png");
R.plot_repeat(static, path=reports_dir / "reid_repeat.png");
R.plot_composition(static, "ethnicity", path=reports_dir / "reid_ethnicity.png");

**Rare stays after suppression, by scope.** Stays in cells smaller than 10, counted within each site (the case when the site is a visible column) and across all sites pooled (when it is not).

In [ ]:
after = R.uniqueness(supp, combos={"all": key})
R.plot_scope(after, path=reports_dir / "reid_scope.png");

**Local suppression on the pooled data (site not released).** Cells are counted across all sites, because the site will not be a column. The within-site version above is kept for comparison. Both plots use the same key: age band, sex, ethnicity group, diagnosis group, outcome and weight band.

In [ ]:
key = ["age_band", "sex", "ethnicity", "diagnosis_group", "died", "weight_band"]
supp_p, rep_pooled = R.suppress_to_k(coarse2, key, ["weight_band", "ethnicity", "diagnosis_group"], by_site=False)
display(rep_pooled)
display(R.uniqueness(supp_p, combos={"all": key}).query("site == 'pooled'"))
R.plot_suppression(rep_pooled, len(coarse2), path=reports_dir / "reid_suppression_pooled.png",
                   title="Pooled data (site not released): rare stays remaining after each field is blanked");
R.plot_suppression(rep2, len(coarse2), path=reports_dir / "reid_suppression_within_site.png",
                   title="Within each site (site released): rare stays remaining after each field is blanked");

**Can the site be guessed?** A classifier tries to predict the site from the coarsened static fields (the Tier 2 set). Accuracy close to the baseline of always guessing the largest site means the fields do not give the site away. The second table shows the accuracy with each column left out.

**Site-inference test: method.** The question is how well the site can be guessed from the columns that would be released. A gradient-boosting classifier (scikit-learn `HistGradientBoostingClassifier`, 100 iterations, default settings, not tuned) is trained to predict the site (GOSH, RBH or SMH) from the columns. Categorical and boolean columns are one-hot encoded, and a missing value is its own category, because how missing values are coded can itself give the site away. Predictions are made out of fold, with 5-fold stratified cross-validation, so every stay is predicted by a model that did not see it; all stays of a patient (site and patient identifier) are kept in the same fold, so repeat admissions cannot leak between training and test. The result is reported as accuracy against the baseline of always guessing the largest site (about 40% in this cohort, not 33%, because the sites differ in size), and as recall for each site, since accuracy alone hides a small site that is always missed. To see which columns give the site away, the accuracy is recomputed with each column left out in turn.

Limits. The classifier is untuned and uses only the columns given, so the accuracy is a lower bound on what a determined user with a stronger model, or outside knowledge of each hospital, could achieve; accuracy near the baseline is encouraging but does not prove that the site cannot be inferred. The test uses stays as the unit and says nothing about re-identifying a patient, which is a separate question covered by the small-cell checks. Only accuracies and recalls are reported; no values from the columns are shown.

In [ ]:
X = supp_p[["age_band", "sex", "ethnicity", "diagnosis_group", "died", "weight_band"]]
groups = static["site"].astype(str) + ":" + static["patient_id"].astype(str)
res_static = R.site_inference(X, supp_p["site"], groups=groups.loc[supp_p.index])
print("accuracy", res_static["accuracy"], "baseline", res_static["baseline"])
display(res_static["recall_by_site"])
display(res_static["accuracy_without"])

In [ ]:
X2 = supp_p[["age_band", "sex", "ethnicity", "died", "weight_band"]]
res2 = R.site_inference(X2, supp_p["site"], groups=groups.loc[supp_p.index], drop_one=False)
print(res2["accuracy"], res2["baseline"]); display(res2["recall_by_site"])

**Figure: can the site be guessed?** Left: accuracy with all static fields and with each field left out in turn, against always guessing the largest site. Right: how often each site is guessed correctly, with and without the diagnosis group.

In [ ]:
R.plot_site_inference(res_static, res2, without_label="diagnosis group", path=reports_dir / "reid_site_inference.png");

**Availability.** Which concepts are collected at only some sites, and can the site be guessed from which concepts a stay has? The availability table is computed from the harmonised files, so it reflects what a user of the release can see (window rule, unit filters and rejected rows applied). The tables list the concepts at fewer than three sites and the share of stays whose availability pattern occurs at one site only, with and without the one-site concepts. The classifier is then run on which concepts each stay has, in three versions: all concepts, without the concepts collected at one site only, and without those collected at one or two sites. All stays of a patient are kept in the same fold. Accuracy is compared with always guessing the largest site.

In [ ]:
avail = H.availability_released(harmonised_dir, static)
one_site = R.concept_coverage(avail).query("n_sites == 1")["concept"].tolist()
two_site = R.concept_coverage(avail).query("n_sites == 2")["concept"].tolist()
print(len(one_site), "one-site and", len(two_site), "two-site concepts")
display(R.concept_coverage(avail).query("n_sites < 3"))
print("all concepts:", R.pattern_site_share(avail))
print("without one-site concepts:", R.pattern_site_share(avail.drop(columns=one_site)))

sp = static.assign(stay_key=static["stay_key"].astype(str)).set_index(["site", "stay_key"])["patient_id"].astype(str)
site_a = avail.index.get_level_values("site")
groups_a = np.asarray(site_a.astype(str)) + ":" + sp.reindex(avail.index).to_numpy()
res_av = {}
for name, drop in {"all concepts": [], "without one-site concepts": one_site,
                   "without one- and two-site concepts": one_site + two_site}.items():
    res_av[name] = R.site_inference(avail.drop(columns=drop).reset_index(drop=True), site_a, groups=groups_a, drop_one=False)
    print(name, res_av[name]["accuracy"], "baseline", res_av[name]["baseline"])
R.plot_site_variants(res_av, path=reports_dir / "reid_site_availability_released.png")

**Which concept group gives the site away?** The classifier is run on the concepts shared by all three sites, with one group of concepts removed at a time. A large drop in accuracy means the group carries much of the site signal.

In [ ]:
base_cols = [c for c in avail.columns if c not in one_site + two_site]
by_group = {}
for c in base_cols:
    by_group.setdefault(C.CONCEPTS[c]["group"], []).append(c)
acc_group = {"none removed": res_av["without one- and two-site concepts"]["accuracy"]}
for g, cols in by_group.items():
    Xg = avail[[c for c in base_cols if c not in cols]].reset_index(drop=True)
    acc_group[f"without {g}"] = R.site_inference(Xg, site_a, groups=groups_a, drop_one=False)["accuracy"]
display(pd.Series(acc_group).sort_values())

**Figure: which concept group gives the site away?** Accuracy of guessing the site from the concepts shared by all three sites, with one group removed at a time. The label on each bar is the change from nothing removed.

In [ ]:
R.plot_group_drop(pd.Series(acc_group), res_av["without one- and two-site concepts"]["baseline"],
                  path=reports_dir / "reid_site_group_drop.png");

**Static fields and availability together.** The classifier is run on the coarsened static fields (age band, sex, ethnicity group, diagnosis group, outcome, weight band) combined with the availability of the concepts shared by all three sites. This is closest to what a user of the pooled release can see. The figure compares it with each part alone.

In [ ]:
st_cols = ["age_band", "sex", "ethnicity", "diagnosis_group", "died", "weight_band"]
st = supp_p[st_cols].copy()
st.index = pd.MultiIndex.from_arrays([static["site"].to_numpy(), static["stay_key"].astype(str).to_numpy()], names=["site", "stay_key"])
comb = st.join(avail.drop(columns=one_site + two_site), how="inner")
print(len(comb), "stays;", comb.shape[1], "columns")

site_c = comb.index.get_level_values("site")
groups_c = np.asarray(site_c.astype(str)) + ":" + sp.reindex(comb.index).to_numpy()
res_comb = R.site_inference(comb.reset_index(drop=True), site_c, groups=groups_c, drop_one=False)
print("static + shared concepts", res_comb["accuracy"], "baseline", res_comb["baseline"])
display(res_comb["recall_by_site"])

R.plot_site_variants({"static fields only": res_static,
                      "shared concepts only": res_av["without one- and two-site concepts"],
                      "static + shared concepts": res_comb},
                     title="How well the site can be guessed from the release",
                     path=reports_dir / "reid_site_combined.png");

**Save the report.** Counts only: uniqueness before and after coarsening and suppression, repeat patients, rare categories, site-only values, and the site-inference accuracy for the static fields, the availability pattern and the two together. The file goes to `reports/` and stays in the SDE.

In [ ]:
def _acc(res):
    return pd.DataFrame([{"accuracy": res["accuracy"], "baseline": res["baseline"]}])

D.write_report({
    "Cohort (stays per site)": static.groupby("site").size().rename("stays").to_frame(),
    "Uniqueness, static fields": uniq,
    "Repeat patients": R.repeat_patients(static),
    "Rare categories": R.rare_categories(static),
    "Values at one site only": R.site_only_values(static),
    "Tier 2 candidates, pooled": res[res["site"] == "pooled"][["ethnicity", "combination", "unique_stays", "stays_in_cells_below_10", "share_below_k"]],
    "Suppression, within site": rep2,
    "Suppression, pooled": rep_pooled,
    "Uniqueness after pooled suppression": R.uniqueness(supp_p, combos={"all": key}),
    "Ethnicity groups": R.group_table(static, "ethnicity"),
    "Diagnosis groups and merges": R.merge_table(static, "diagnosis_group"),
    "Site guessed from static fields": _acc(res_static).join(res_static["recall_by_site"].rename("recall").to_frame().T.reset_index(drop=True)),
    "Site guessed from availability": pd.concat([_acc(v).assign(version=k) for k, v in res_av.items()]),
    "Site guessed, static + shared concepts": _acc(res_comb),
    "Accuracy without each concept group": pd.Series(acc_group, name="accuracy").to_frame(),
}, reports_dir / f"Reid_{date.today()}.md", title="Re-identification review")